### Analisis Komparatif Algoritma Klasifikasi: Prediksi Customer Churn
**Studi Kasus:** Mengidentifikasi probabilitas pelanggan berhenti berlangganan (Churn) pada perusahaan layanan.
**Metode:** Random Forest, XGBoost, dan K-Nearest Neighbors (KNN).

#### 1. Preparasi Pustaka dan Dataset
Memuat dependensi analisis data dan mengambil dataset dari repositori publik.

In [ ]:
import pandas as pd                                  # Manipulasi data tabuler
import numpy as np                                   # Operasi vektor dan matriks
import matplotlib.pyplot as plt                      # Render visualisasi dasar
import seaborn as sns                                # Visualisasi statistik tingkat lanjut

from sklearn.model_selection import train_test_split # Split dataset
from sklearn.preprocessing import StandardScaler, LabelEncoder # Normalisasi dan encoding
from sklearn.ensemble import RandomForestClassifier  # Model Random Forest
from sklearn.neighbors import KNeighborsClassifier   # Model KNN
from xgboost import XGBClassifier                    # Model XGBoost
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix # Evaluasi

# Mengambil dataset Telco Churn dari repositori publik (GitHub/Raw)
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)                                # Ekstraksi file CSV ke DataFrame

print(f"Data berhasil dimuat. Dimensi: {df.shape}")  # Validasi ukuran data
df.head()                                            # Inspeksi struktur kolom

#### 2. Pembersihan Data (*Data Cleansing*)
Menghapus kolom yang tidak relevan (ID) dan menangani nilai kosong pada fitur numerik.

In [ ]:
df = df.drop('customerID', axis=1)                   # Menghapus ID karena tidak memiliki nilai prediktif

# Konversi TotalCharges ke numerik (menangani spasi kosong sebagai NaN)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()                                     # Menghapus baris dengan nilai kosong (missing values)

# Encoding label target (Churn: Yes/No -> 1/0)
df['Churn'] = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

print("Pembersihan selesai. Data siap di-encode.")

#### 3. Rekayasa Fitur (*Feature Engineering*)
Mengubah data kategorikal (teks) menjadi angka agar dapat diproses oleh algoritma Machine Learning.

In [ ]:
# Mengubah kolom kategorikal menjadi variabel dummy (One-Hot Encoding)
df_final = pd.get_dummies(df)

X = df_final.drop('Churn', axis=1)                   # Memisahkan fitur independen
y = df_final['Churn']                                # Memisahkan variabel target

# Pembagian data: 80% Latih, 20% Uji
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardisasi data khusus untuk algoritma KNN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Preprocessing selesai. Fitur telah di-encode dan di-scale.")

#### 4. Pelatihan Model (*Model Training*)
Eksekusi tiga algoritma dengan parameter standar industri.

In [ ]:
# Inisialisasi arsitektur model
rf = RandomForestClassifier(n_estimators=100, random_state=42)
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
knn = KNeighborsClassifier(n_neighbors=5)

# Proses pembelajaran
rf.fit(X_train, y_train)                             # Training Random Forest
xgb.fit(X_train, y_train)                            # Training XGBoost
knn.fit(X_train_scaled, y_train)                     # Training KNN (menggunakan data terskala)

print("Ketiga model telah berhasil dilatih.")

#### 5. Evaluasi Komparatif
Membandingkan presisi dan akurasi tiap model dalam mendeteksi pelanggan yang akan Churn.

In [ ]:
models = {'Random Forest': rf, 'XGBoost': xgb, 'KNN': knn}
results = {}

for name, model in models.items():
    xt = X_test_scaled if name == 'KNN' else X_test
    y_pred = model.predict(xt)
    results[name] = y_pred
    
    print(f"\n=== {name} ===")
    print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred))

#### 6. Visualisasi Diagnostik
Menampilkan Confusion Matrix untuk melihat detail prediksi benar/salah pada data uji.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for i, (name, y_pred) in enumerate(results.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=axes[i])
    axes[i].set_title(f'Confusion Matrix: {name}')
    axes[i].set_xlabel('Prediksi')
    axes[i].set_ylabel('Aktual')

plt.tight_layout()
plt.show()